# Standalone dataset preparation for Task 2

This notebook builds the merged three-class YOLO dataset used by the repository's Task 2 detector:

| Class ID | Class | Source |
|---:|---|---|
| 0 | cone | nuScenes v1.0-mini |
| 1 | barrier | nuScenes v1.0-mini |
| 2 | stop_sign | COCO 2017 and optional pseudo-labels on nuScenes |

It reads a local nuScenes copy, downloads COCO annotations and only the COCO images that contain stop signs, converts both sources into YOLO labels, combines them in one train/val tree, and writes a ready-to-use `data.yaml`. It also writes nuScenes camera/depth sidecars, matching the repo's Task 1 conversion so they remain available for Task 3.

The notebook is self-contained and does not import code from the repository. By default, outputs and the COCO cache go under `~/datasets/`, outside the repository folder. Edit the paths below before running all cells. The nuScenes path can be the dataroot itself or a parent folder containing it.

The COCO image download is a stop-sign subset, not the full COCO image archive. Pseudo-labeling is enabled because nuScenes does not annotate stop signs; it helps avoid treating visible signs in nuScenes images as background.

In [ ]:
%pip install nuscenes-devkit ultralytics requests tqdm pyyaml pyquaternion pillow

In [ ]:
from pathlib import Path

# ---- edit these paths/settings ----
NUSCENES_PATH = Path(r"C:\path\to\nuscenes-mini")  # local copy; wrapper folder is OK
OUTPUT_ROOT = Path.home() / "datasets" / "eric_task2_merged_yolo"
COCO_CACHE = Path.home() / "datasets" / "eric_task2_coco_cache"

# Same validation scenes as configs/paths.yaml in this repo.
VAL_SCENES = {"scene-0655", "scene-0553"}

# nuScenes has no stop-sign labels. Set False only if you deliberately want to
# skip the repo's COCO-pretrained stop-sign pseudo-label step.
ADD_NUSCENES_STOP_SIGN_PSEUDO_LABELS = True
PSEUDO_WEIGHTS = "yolo11n.pt"  # COCO-pretrained Ultralytics model
PSEUDO_CONFIDENCE = 0.5
PSEUDO_IMAGE_SIZE = 960
PSEUDO_BATCH_SIZE = 16
PSEUDO_DEVICE = None  # None = Ultralytics auto-selects GPU/CPU; e.g. "cpu" or 0
# ----------------------------------

print("nuScenes input:", NUSCENES_PATH)
print("merged output :", OUTPUT_ROOT)
print("COCO cache    :", COCO_CACHE)

## 1. Imports, paths, and output folders

The path discovery accepts up to three wrapper-folder levels. Images are hard-linked where possible, with a file-copy fallback when the filesystem does not support hard links.

In [ ]:
import json
import os
import shutil
from collections import Counter

import numpy as np
from PIL import Image, ImageDraw
from tqdm.auto import tqdm


def find_nuscenes_root(path, max_depth=3):
    path = Path(path).expanduser().resolve()
    if not path.is_dir():
        raise FileNotFoundError(f"nuScenes path does not exist: {path}")
    frontier = [path]
    for depth in range(max_depth + 1):
        for candidate in frontier:
            if (candidate / "samples").is_dir() and (candidate / "v1.0-mini").is_dir():
                return candidate
        if depth == max_depth:
            break
        frontier = sorted(
            child for parent in frontier for child in parent.iterdir() if child.is_dir()
        )
    raise FileNotFoundError(
        f"Could not find a folder containing both samples/ and v1.0-mini/ under {path}"
    )


def stage_image(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return "present"
    try:
        os.link(source, destination)
        return "hardlink"
    except (OSError, NotImplementedError):
        shutil.copy2(source, destination)
        return "copy"


NUSCENES_ROOT = find_nuscenes_root(NUSCENES_PATH)
OUTPUT_ROOT = OUTPUT_ROOT.expanduser().resolve()
COCO_CACHE = COCO_CACHE.expanduser().resolve()
if OUTPUT_ROOT == NUSCENES_ROOT or NUSCENES_ROOT in OUTPUT_ROOT.parents:
    raise ValueError("Choose an output folder outside the nuScenes source tree")

for split in ("train", "val"):
    for subfolder in ("images", "labels", "meta"):
        (OUTPUT_ROOT / subfolder / split).mkdir(parents=True, exist_ok=True)
COCO_CACHE.mkdir(parents=True, exist_ok=True)

print("Using nuScenes dataroot:", NUSCENES_ROOT)
print("Writing merged dataset to:", OUTPUT_ROOT)

## 2. Convert nuScenes to YOLO

This follows the repository converter: all six cameras, split by scene, class IDs 0/1, visibility level at least 2, at least one lidar/radar point, projected cuboid corners in front of the camera, and boxes at least 4 pixels wide and high. Each nuScenes image also gets a JSON sidecar with camera intrinsics, camera height/pitch, and box depth.

In [ ]:
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.geometry_utils import BoxVisibility, view_points

CAMERAS = [
    "CAM_FRONT", "CAM_FRONT_LEFT", "CAM_FRONT_RIGHT",
    "CAM_BACK", "CAM_BACK_LEFT", "CAM_BACK_RIGHT",
]
IMAGE_WIDTH, IMAGE_HEIGHT = 1600, 900
MIN_BOX_PIXELS = 4
MIN_VISIBILITY = 2
NUSCENES_CLASS = {
    "movable_object.trafficcone": 0,
    "movable_object.barrier": 1,
}
CLASS_NAMES = ["cone", "barrier", "stop_sign"]


def camera_height_pitch(nusc, sample_data):
    calibrated = nusc.get("calibrated_sensor", sample_data["calibrated_sensor_token"])
    optical_axis = Quaternion(calibrated["rotation"]).rotation_matrix @ np.array([0.0, 0.0, 1.0])
    pitch = float(np.arcsin(np.clip(-optical_axis[2], -1.0, 1.0)))
    return float(calibrated["translation"][2]), pitch


# Clear only prior nuScenes outputs, identified by their meta sidecars.
# COCO files have no sidecar and are left in place.
for split in ("train", "val"):
    meta_dir = OUTPUT_ROOT / "meta" / split
    for meta_file in list(meta_dir.glob("*.json")):
        (OUTPUT_ROOT / "labels" / split / f"{meta_file.stem}.txt").unlink(missing_ok=True)
        for extension in (".jpg", ".jpeg", ".png"):
            (OUTPUT_ROOT / "images" / split / f"{meta_file.stem}{extension}").unlink(missing_ok=True)
        meta_file.unlink(missing_ok=True)

nusc = NuScenes(version="v1.0-mini", dataroot=str(NUSCENES_ROOT), verbose=False)
scene_names = {scene["name"] for scene in nusc.scene}
unknown_val_scenes = VAL_SCENES - scene_names
if unknown_val_scenes:
    raise ValueError(f"Validation scenes not found in this nuScenes copy: {sorted(unknown_val_scenes)}")

nuscenes_stats = {
    "images": Counter(),
    "boxes": {"train": Counter(), "val": Counter()},
    "dropped": Counter(),
    "scenes": {"train": [], "val": []},
    "staging": Counter(),
}

for scene in tqdm(nusc.scene, desc="nuScenes scenes"):
    split = "val" if scene["name"] in VAL_SCENES else "train"
    nuscenes_stats["scenes"][split].append(scene["name"])
    sample_token = scene["first_sample_token"]
    while sample_token:
        sample = nusc.get("sample", sample_token)
        for camera in CAMERAS:
            if camera not in sample["data"]:
                nuscenes_stats["dropped"][f"missing_{camera}"] += 1
                continue

            sample_data_token = sample["data"][camera]
            sample_data = nusc.get("sample_data", sample_data_token)
            image_path, boxes, camera_intrinsics = nusc.get_sample_data(
                sample_data_token, box_vis_level=BoxVisibility.ANY
            )
            camera_height, camera_pitch = camera_height_pitch(nusc, sample_data)
            yolo_lines, metadata_objects = [], []

            for box in boxes:
                if box.name not in NUSCENES_CLASS:
                    continue
                annotation = nusc.get("sample_annotation", box.token)
                visibility = int(annotation["visibility_token"])
                if visibility < MIN_VISIBILITY:
                    nuscenes_stats["dropped"]["low_visibility"] += 1
                    continue
                if annotation["num_lidar_pts"] + annotation["num_radar_pts"] == 0:
                    nuscenes_stats["dropped"]["no_lidar_radar"] += 1
                    continue

                corners = box.corners()
                if corners[2].min() < 0.1:
                    nuscenes_stats["dropped"]["behind_camera"] += 1
                    continue

                image_corners = view_points(corners, camera_intrinsics, normalize=True)[:2]
                x1, y1 = image_corners.min(axis=1)
                x2, y2 = image_corners.max(axis=1)
                truncated = bool(x1 < 0 or y1 < 0 or x2 > IMAGE_WIDTH or y2 > IMAGE_HEIGHT)
                x1, x2 = np.clip([x1, x2], 0, IMAGE_WIDTH)
                y1, y2 = np.clip([y1, y2], 0, IMAGE_HEIGHT)
                box_width, box_height = x2 - x1, y2 - y1
                if box_width < MIN_BOX_PIXELS or box_height < MIN_BOX_PIXELS:
                    nuscenes_stats["dropped"]["too_small"] += 1
                    continue

                class_id = NUSCENES_CLASS[box.name]
                yolo_lines.append(
                    f"{class_id} {(x1 + x2) / (2 * IMAGE_WIDTH):.6f} "
                    f"{(y1 + y2) / (2 * IMAGE_HEIGHT):.6f} "
                    f"{box_width / IMAGE_WIDTH:.6f} {box_height / IMAGE_HEIGHT:.6f}"
                )
                metadata_objects.append({
                    "cls": int(class_id),
                    "box": [float(x1), float(y1), float(x2), float(y2)],
                    "z": float(box.center[2]),
                    "dims_whl": [float(value) for value in box.wlh],
                    "trunc": truncated,
                    "vis": visibility,
                })
                nuscenes_stats["boxes"][split][class_id] += 1

            image_path = Path(image_path)
            destination = OUTPUT_ROOT / "images" / split / image_path.name
            nuscenes_stats["staging"][stage_image(image_path, destination)] += 1
            (OUTPUT_ROOT / "labels" / split / f"{image_path.stem}.txt").write_text(
                "\n".join(yolo_lines), encoding="utf-8"
            )
            sidecar = {
                "image": image_path.name,
                "camera": camera,
                "scene": scene["name"],
                "width": IMAGE_WIDTH,
                "height": IMAGE_HEIGHT,
                "K": np.asarray(camera_intrinsics).tolist(),
                "h_cam": camera_height,
                "pitch": camera_pitch,
                "objects": metadata_objects,
            }
            (OUTPUT_ROOT / "meta" / split / f"{image_path.stem}.json").write_text(
                json.dumps(sidecar), encoding="utf-8"
            )
            nuscenes_stats["images"][split] += 1
        sample_token = sample["next"]

print("nuScenes images:", dict(nuscenes_stats["images"]))
print("nuScenes boxes :", {s: dict(c) for s, c in nuscenes_stats["boxes"].items()})
print("Dropped boxes  :", dict(nuscenes_stats["dropped"]))
print("Scene split    :", nuscenes_stats["scenes"])

## 3. Download COCO stop-sign images and convert labels

The COCO annotations archive contains both 2017 instance files. This step keeps only images with non-crowd `stop sign` annotations and downloads those images, writing each box as class 2 in normalized YOLO `class cx cy width height` form.

In [ ]:
import zipfile
import requests
from concurrent.futures import ThreadPoolExecutor

COCO_ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
COCO_IMAGE_URL = "http://images.cocodataset.org/{split}/{file_name}"
COCO_STOP_SIGN_CLASS = 2
COCO_SPLITS = {"train2017": "train", "val2017": "val"}
ANNOTATIONS_ZIP = COCO_CACHE / "annotations_trainval2017.zip"


def download_file(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    partial = destination.with_name(destination.name + ".part")
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with partial.open("wb") as output_file:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    output_file.write(chunk)
    partial.replace(destination)
    return destination


def load_coco_annotations(split_name):
    annotation_path = COCO_CACHE / f"instances_{split_name}.json"
    if annotation_path.exists():
        return json.loads(annotation_path.read_text(encoding="utf-8"))
    download_file(COCO_ANNOTATIONS_URL, ANNOTATIONS_ZIP)
    expected_name = f"annotations/instances_{split_name}.json"
    with zipfile.ZipFile(ANNOTATIONS_ZIP) as archive:
        member = next((name for name in archive.namelist() if name.endswith(expected_name)), None)
        if member is None:
            raise FileNotFoundError(f"{expected_name} not found in COCO annotation archive")
        annotation_path.write_bytes(archive.read(member))
    return json.loads(annotation_path.read_text(encoding="utf-8"))


def coco_stop_sign_records(annotation_data):
    category_ids = {
        category["id"] for category in annotation_data["categories"]
        if category["name"] == "stop sign"
    }
    if not category_ids:
        raise ValueError("COCO annotation file has no 'stop sign' category")
    records = {}
    for annotation in annotation_data["annotations"]:
        if annotation["category_id"] not in category_ids or annotation.get("iscrowd", 0):
            continue
        records.setdefault(annotation["image_id"], {"boxes": []})["boxes"].append(annotation["bbox"])
    images_by_id = {image["id"]: image for image in annotation_data["images"]}
    for image_id, record in records.items():
        record["info"] = images_by_id[image_id]
    return records


def fetch_coco_image(item):
    file_name, split_name, destination = item
    if destination.exists() and destination.stat().st_size > 0:
        return True
    url = COCO_IMAGE_URL.format(split=split_name, file_name=file_name)
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(response.content)
        return True
    except requests.RequestException as error:
        return False


def coco_yolo_lines(record):
    image_width = record["info"]["width"]
    image_height = record["info"]["height"]
    lines = []
    for x, y, box_width, box_height in record["boxes"]:
        if box_width < 2 or box_height < 2:
            continue
        lines.append(
            f"{COCO_STOP_SIGN_CLASS} {(x + box_width / 2) / image_width:.6f} "
            f"{(y + box_height / 2) / image_height:.6f} "
            f"{box_width / image_width:.6f} {box_height / image_height:.6f}"
        )
    return lines


coco_stats = {}
for coco_split, yolo_split in COCO_SPLITS.items():
    annotation_data = load_coco_annotations(coco_split)
    records = coco_stop_sign_records(annotation_data)
    print(f"{coco_split}: {len(records)} images have stop-sign annotations")

    work = []
    for record in records.values():
        file_name = record["info"]["file_name"]
        destination = OUTPUT_ROOT / "images" / yolo_split / file_name
        work.append((file_name, coco_split, destination))

    with ThreadPoolExecutor(max_workers=16) as pool:
        downloaded = list(tqdm(pool.map(fetch_coco_image, work), total=len(work), desc=f"COCO {yolo_split} images"))

    successful = 0
    boxes_written = 0
    for record in records.values():
        file_name = record["info"]["file_name"]
        image_path = OUTPUT_ROOT / "images" / yolo_split / file_name
        if not image_path.exists():
            continue
        lines = coco_yolo_lines(record)
        (OUTPUT_ROOT / "labels" / yolo_split / f"{Path(file_name).stem}.txt").write_text(
            "\n".join(lines), encoding="utf-8"
        )
        successful += 1
        boxes_written += len(lines)

    coco_stats[yolo_split] = {
        "annotated_images": len(records),
        "download_success": sum(downloaded),
        "images_written": successful,
        "boxes_written": boxes_written,
    }
    print(yolo_split, coco_stats[yolo_split])

## 4. Add COCO-pretrained stop-sign pseudo-labels to nuScenes

The repo does this because nuScenes has cone/barrier labels but no stop-sign labels. A COCO-pretrained detector predicts only COCO class index 11 (`stop sign`); detections at confidence 0.5 or higher are added as unified class 2. This cell processes only images with nuScenes metadata sidecars and replaces any previous class-2 pseudo-label lines before writing fresh ones.

In [ ]:
from ultralytics import YOLO


def nuscenes_images_for_split(split):
    found = []
    for meta_path in sorted((OUTPUT_ROOT / "meta" / split).glob("*.json")):
        for extension in (".jpg", ".png", ".jpeg"):
            candidate = OUTPUT_ROOT / "images" / split / f"{meta_path.stem}{extension}"
            if candidate.exists():
                found.append(candidate)
                break
    return found


pseudo_stats = {}
if ADD_NUSCENES_STOP_SIGN_PSEUDO_LABELS:
    model = YOLO(PSEUDO_WEIGHTS)
    COCO_PRETRAINED_STOP_SIGN_INDEX = 11

    for split in ("train", "val"):
        images = nuscenes_images_for_split(split)
        added_images = 0
        added_boxes = 0
        for start in tqdm(range(0, len(images), PSEUDO_BATCH_SIZE), desc=f"Pseudo-label {split}"):
            batch_paths = images[start : start + PSEUDO_BATCH_SIZE]
            predictions = model.predict(
                [str(image_path) for image_path in batch_paths],
                imgsz=PSEUDO_IMAGE_SIZE,
                conf=PSEUDO_CONFIDENCE,
                device=PSEUDO_DEVICE,
                classes=[COCO_PRETRAINED_STOP_SIGN_INDEX],
                verbose=False,
            )
            for image_path, prediction in zip(batch_paths, predictions):
                label_path = OUTPUT_ROOT / "labels" / split / f"{image_path.stem}.txt"
                existing = label_path.read_text(encoding="utf-8").splitlines() if label_path.exists() else []
                kept_lines = [line for line in existing if line.strip() and int(line.split()[0]) != 2]
                height, width = prediction.orig_shape
                new_lines = []
                for box in prediction.boxes:
                    x1, y1, x2, y2 = [float(value) for value in box.xyxy[0]]
                    box_width, box_height = x2 - x1, y2 - y1
                    if box_width < 4 or box_height < 4:
                        continue
                    new_lines.append(
                        f"2 {(x1 + x2) / (2 * width):.6f} {(y1 + y2) / (2 * height):.6f} "
                        f"{box_width / width:.6f} {box_height / height:.6f}"
                    )
                label_path.write_text("\n".join(kept_lines + new_lines), encoding="utf-8")
                added_boxes += len(new_lines)
                added_images += bool(new_lines)
        pseudo_stats[split] = {"images_with_stop_signs": added_images, "boxes": added_boxes}
        print(split, pseudo_stats[split])
else:
    print("Skipped. nuScenes images will remain unlabeled for stop signs.")

## 5. Validate and write `data.yaml`

This combines the two sources by putting both into the same `images/{train,val}` and `labels/{train,val}` folders. The YAML uses absolute paths so Ultralytics can read the dataset from any working directory.

In [ ]:
from collections import Counter
import yaml

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
merge_report = {}
problems = []

for split in ("train", "val"):
    image_dir = OUTPUT_ROOT / "images" / split
    label_dir = OUTPUT_ROOT / "labels" / split
    images = sorted(path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
    counts = Counter()
    missing_labels = []
    malformed = []
    backgrounds = 0

    for image_path in images:
        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            missing_labels.append(image_path.name)
            continue
        lines = [line for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]
        if not lines:
            backgrounds += 1
        for line in lines:
            parts = line.split()
            if len(parts) != 5:
                malformed.append(f"{label_path.name}: {line}")
                continue
            try:
                class_id = int(parts[0])
                values = [float(value) for value in parts[1:]]
            except ValueError:
                malformed.append(f"{label_path.name}: {line}")
                continue
            if class_id not in (0, 1, 2) or not all(0 <= value <= 1 for value in values) or values[2] <= 0 or values[3] <= 0:
                malformed.append(f"{label_path.name}: {line}")
                continue
            counts[class_id] += 1

    orphan_labels = [
        path.name for path in label_dir.glob("*.txt")
        if not any((image_dir / f"{path.stem}{extension}").exists() for extension in IMAGE_EXTENSIONS)
    ]
    merge_report[split] = {
        "images": len(images),
        "backgrounds": backgrounds,
        "instances": {CLASS_NAMES[class_id]: counts[class_id] for class_id in range(3)},
        "missing_labels": missing_labels,
        "orphan_labels": orphan_labels,
        "malformed_labels": malformed,
    }
    print(
        f"{split}: {len(images)} images; "
        + ", ".join(f"{CLASS_NAMES[class_id]}={counts[class_id]}" for class_id in range(3))
    )
    if missing_labels:
        problems.append(f"{split}: {len(missing_labels)} images have no label file")
    if orphan_labels:
        problems.append(f"{split}: {len(orphan_labels)} label files have no image")
    if malformed:
        problems.append(f"{split}: {len(malformed)} malformed YOLO label lines")
    for class_id in range(3):
        if counts[class_id] == 0:
            problems.append(f"{split}: no {CLASS_NAMES[class_id]} instances")

DATA_YAML = OUTPUT_ROOT / "data.yaml"
data_config = {
    "path": OUTPUT_ROOT.as_posix(),
    "train": (OUTPUT_ROOT / "images" / "train").as_posix(),
    "val": (OUTPUT_ROOT / "images" / "val").as_posix(),
    "names": {class_id: class_name for class_id, class_name in enumerate(CLASS_NAMES)},
}
DATA_YAML.write_text(yaml.safe_dump(data_config, sort_keys=False), encoding="utf-8")
(OUTPUT_ROOT / "dataset_report.json").write_text(
    json.dumps({"splits": merge_report, "nuscenes": {
        "images": dict(nuscenes_stats["images"]),
        "boxes": {split: dict(counts) for split, counts in nuscenes_stats["boxes"].items()},
        "scenes": nuscenes_stats["scenes"],
        "dropped": dict(nuscenes_stats["dropped"]),
    }, "coco": coco_stats, "pseudo_labels": pseudo_stats}, indent=2),
    encoding="utf-8",
)

print("\nUltralytics dataset YAML:", DATA_YAML)
print(DATA_YAML.read_text(encoding="utf-8"))
if problems:
    print("Review these dataset checks:")
    for problem in problems:
        print(" -", problem)
else:
    print("All image/label pairing and class-presence checks passed.")

## 6. Preview a few merged labels

This displays a few train/val images with their YOLO boxes and class names, providing a quick visual check before training.

In [ ]:
import random
from IPython.display import display

random.seed(7)
for split in ("train", "val"):
    image_paths = sorted(
        path for path in (OUTPUT_ROOT / "images" / split).iterdir()
        if path.suffix.lower() in IMAGE_EXTENSIONS
        and (OUTPUT_ROOT / "labels" / split / f"{path.stem}.txt").exists()
    )
    random.shuffle(image_paths)
    print(f"{split} examples")
    shown = 0
    for image_path in image_paths:
        label_path = OUTPUT_ROOT / "labels" / split / f"{image_path.stem}.txt"
        lines = [line.split() for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]
        if not lines:
            continue
        image = Image.open(image_path).convert("RGB")
        draw = ImageDraw.Draw(image)
        width, height = image.size
        for parts in lines:
            class_id = int(parts[0])
            cx, cy, box_width, box_height = map(float, parts[1:])
            x1 = (cx - box_width / 2) * width
            y1 = (cy - box_height / 2) * height
            x2 = (cx + box_width / 2) * width
            y2 = (cy + box_height / 2) * height
            draw.rectangle((x1, y1, x2, y2), outline="lime", width=3)
            draw.text((x1, max(0, y1 - 16)), CLASS_NAMES[class_id], fill="lime")
        display(image.resize((800, 450)))
        shown += 1
        if shown == 3:
            break

print("\nDataset ready at:", OUTPUT_ROOT)
print("Use this file for Task 2 training:", DATA_YAML)